# 🏆 Modelo Predictivo v2 — Mundial FIFA 2026
### Red Neuronal Ensemble + Jugadores + Eliminatorias + Resultados Actuales

**Arquitectura:**
- **MLP Classifier** (256-128-64-32 neuronas, ReLU, Adam)
- **GradientBoosting Classifier** (200 estimators, lr=0.05)
- **Ensemble** (promedio de probabilidades)
- **Regressores Poisson** (GradientBoosting λ_A, λ_B)

**Features (35):** ranking, eliminatorias, combinatoria de jugadores, valor de mercado, edad, forma WC

**Métricas Ensemble:** Acc=62.5% · F1=0.621 · AUC=0.810

## 1. Importaciones y Configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)
import pickle
from scipy.stats import poisson

# Colores
C_WIN  = '#56d364'
C_DRAW = '#d29922'
C_LOSE = '#f85149'
C_BLUE = '#58a6ff'
C_BG   = '#0d1117'
C_PANEL= '#161b22'

plt.rcParams.update({
    'figure.facecolor': C_BG, 'axes.facecolor': C_PANEL,
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#e6edf3',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e',
    'text.color': '#e6edf3', 'grid.color': '#30363d',
    'grid.linewidth': 0.5
})

print("✅ Librerías cargadas correctamente")
print(f"   numpy {np.__version__} | pandas {pd.__version__}")

## 2. Dataset: 48 Equipos con 16 variables

In [ ]:
# Formato: rank, conf_idx, qual_gf, qual_ga, qual_w, qual_m,
#          top_scorer_goals, squad_rating, market_value_M, avg_age,
#          xg_qual, injury_factor, wc_pts, wc_gf, wc_ga, wc_played
CONF_IDX = {'UEFA':1,'CONMEBOL':2,'CONCACAF':3,'CAF':4,'AFC':5,'OFC':6}

TEAMS = {
"Francia":        [1,1,3.20,0.80,0.80,10,9,88.2,1050,26.1,3.40,0.92,0,0,0,0],
"España":         [2,1,3.00,1.00,0.70,10,7,87.5,980,26.5,3.10,0.95,0,0,0,0],
"Argentina":      [3,2,1.94,0.94,0.67,18,8,87.0,760,28.0,2.10,0.93,0,0,0,0],
"Inglaterra":     [4,1,3.00,0.90,0.80,10,7,86.5,1100,25.8,3.20,0.90,0,0,0,0],
"Portugal":       [5,1,2.80,0.70,0.90,10,7,85.8,920,27.5,2.90,0.94,0,0,0,0],
"Brasil":         [6,2,1.61,1.22,0.39,18,5,86.0,1200,26.5,1.80,0.91,0,0,0,0],
"Países Bajos":   [7,1,2.90,1.40,0.70,10,8,84.5,870,26.2,2.80,0.93,0,0,0,0],
"Marruecos":      [8,4,2.50,0.50,0.83,6,4,80.0,320,25.8,2.20,0.95,0,0,0,0],
"Bélgica":        [9,1,2.70,1.20,0.70,10,6,83.5,680,29.5,2.60,0.88,0,0,0,0],
"Alemania":       [10,1,3.30,1.10,0.80,10,9,84.0,890,24.8,3.40,0.92,0,0,0,0],
"Croacia":        [11,1,2.20,1.10,0.60,10,5,82.5,420,30.2,2.00,0.90,0,0,0,0],
"Colombia":       [13,2,1.44,1.22,0.44,18,7,82.0,480,25.5,1.55,0.93,0,0,0,0],
"Senegal":        [14,4,2.00,0.83,0.67,6,4,79.5,280,25.0,1.80,0.94,0,0,0,0],
"México":         [15,3,1.86,1.14,0.50,14,5,80.5,390,27.5,1.90,0.90,3,2,0,1],
"Estados Unidos": [16,3,2.00,1.00,0.57,14,8,80.0,520,25.0,2.10,0.93,0,0,0,0],
"Uruguay":        [17,2,1.56,1.11,0.44,18,6,81.0,430,28.5,1.65,0.91,0,0,0,0],
"Japón":          [18,5,2.67,0.56,0.78,9,9,81.5,560,25.5,2.50,0.96,0,0,0,0],
"Suiza":          [19,1,2.50,0.80,0.70,10,5,80.5,450,28.0,2.30,0.93,0,0,0,0],
"Irán":           [21,5,1.67,1.00,0.61,18,7,77.5,180,27.5,1.50,0.91,0,0,0,0],
"Turquía":        [22,1,2.40,1.40,0.60,10,7,79.5,490,26.8,2.20,0.92,0,0,0,0],
"Ecuador":        [23,2,1.44,0.94,0.44,18,5,79.0,310,25.8,1.45,0.93,0,0,0,0],
"Austria":        [24,1,2.50,1.40,0.70,10,6,79.5,410,26.2,2.30,0.94,0,0,0,0],
"Corea del Sur":  [25,5,1.78,1.11,0.56,9,6,79.0,380,26.5,1.65,0.93,3,2,1,1],
"Australia":      [27,5,1.44,1.33,0.44,9,5,77.5,220,28.0,1.30,0.92,0,0,0,0],
"Argelia":        [28,4,2.00,0.83,0.67,6,4,77.0,160,27.5,1.70,0.93,0,0,0,0],
"Egipto":         [29,4,2.00,0.67,0.67,6,4,76.5,140,27.8,1.65,0.92,0,0,0,0],
"Canadá":         [30,3,1.79,0.86,0.57,14,8,78.0,340,25.2,1.85,0.93,0,0,0,0],
"Noruega":        [31,1,3.50,1.20,0.80,10,16,79.5,560,25.8,3.20,0.93,0,0,0,0],
"Panamá":         [33,3,1.29,1.29,0.36,14,4,73.5,95,27.5,1.10,0.94,0,0,0,0],
"Costa de Marfil":[34,4,1.67,1.00,0.50,6,3,74.5,210,27.8,1.40,0.91,0,0,0,0],
"Suecia":         [38,1,2.20,1.40,0.50,10,8,76.0,360,26.5,1.90,0.92,0,0,0,0],
"Paraguay":       [40,2,1.17,1.44,0.28,18,5,73.5,150,27.0,1.05,0.90,0,0,0,0],
"República Checa":[41,1,1.80,1.20,0.40,10,5,74.0,220,28.5,1.60,0.93,0,0,0,1],
"Escocia":        [43,1,2.00,1.60,0.40,10,5,72.5,180,28.0,1.75,0.92,0,0,0,0],
"Túnez":          [44,4,1.50,1.17,0.50,6,3,71.5,120,27.5,1.25,0.91,0,0,0,0],
"RD Congo":       [46,4,1.33,0.83,0.50,6,3,70.5,95,27.0,1.05,0.92,0,0,0,0],
"Uzbekistán":     [50,5,1.50,1.00,0.50,9,4,70.0,85,26.5,1.20,0.93,0,0,0,0],
"Qatar":          [55,5,1.11,1.94,0.28,9,3,68.5,75,27.0,0.90,0.91,0,0,0,0],
"Irak":           [57,5,1.14,1.43,0.36,7,4,68.0,65,27.0,0.85,0.92,0,0,0,0],
"Sudáfrica":      [60,4,1.67,1.17,0.50,6,3,67.5,80,28.0,1.35,0.93,0,0,0,1],
"Arabia Saudita": [61,5,1.44,1.78,0.39,9,5,68.0,110,27.5,1.20,0.91,0,0,0,0],
"Jordania":       [63,5,1.17,1.67,0.33,6,4,66.5,55,26.5,0.90,0.92,0,0,0,0],
"Bosnia y Herz.": [65,1,1.40,1.80,0.30,10,3,69.0,95,29.0,1.15,0.90,0,0,0,0],
"Cabo Verde":     [69,4,1.33,1.17,0.50,6,2,66.0,55,27.0,1.00,0.93,0,0,0,0],
"Ghana":          [74,4,1.33,1.50,0.33,6,3,65.0,70,27.0,1.00,0.91,0,0,0,0],
"Curazao":        [82,3,0.75,1.25,0.25,8,2,61.5,30,27.5,0.60,0.92,0,0,0,0],
"Haití":          [83,3,0.63,1.50,0.13,8,2,60.5,25,27.0,0.50,0.91,0,0,0,0],
"Nueva Zelanda":  [85,6,3.50,1.88,0.88,8,9,60.0,40,27.5,2.80,0.94,0,0,0,0],
}

PLAYERS = {
"Francia":        [("Kylian Mbappé",9,93,180,"DEL"),("Antoine Griezmann",5,87,25,"CAM"),("A. Tchouaméni",2,86,90,"MCD")],
"España":         [("Lamine Yamal",7,88,180,"EXT"),("Pedri",4,88,120,"CAM"),("Álvaro Morata",5,85,35,"DEL")],
"Argentina":      [("Lionel Messi",8,93,20,"DEL"),("Julián Álvarez",6,88,90,"DEL"),("Rodrigo De Paul",3,85,35,"MCI")],
"Inglaterra":     [("Harry Kane",7,90,80,"DEL"),("Jude Bellingham",7,91,180,"CAM"),("Phil Foden",5,88,130,"EXT")],
"Portugal":       [("Cristiano Ronaldo",7,90,15,"DEL"),("Bruno Fernandes",5,87,60,"CAM"),("Bernardo Silva",4,88,80,"CAM")],
"Brasil":         [("Vinicius Jr.",5,92,200,"EXT"),("Rodrygo",4,87,120,"EXT"),("Raphinha",4,86,80,"EXT")],
"Países Bajos":   [("Cody Gakpo",8,86,80,"EXT"),("Memphis Depay",5,85,18,"DEL"),("Frenkie de Jong",3,87,65,"MCI")],
"Marruecos":      [("Y. En-Nesyri",4,83,30,"DEL"),("Hakim Ziyech",3,82,12,"EXT"),("Achraf Hakimi",2,87,60,"LAT")],
"Bélgica":        [("Romelu Lukaku",6,86,22,"DEL"),("Kevin De Bruyne",4,91,35,"CAM"),("Lois Openda",5,84,50,"DEL")],
"Alemania":       [("Kai Havertz",9,86,65,"CAM"),("Florian Wirtz",7,90,150,"CAM"),("Jamal Musiala",5,89,120,"EXT")],
"Corea del Sur":  [("Son Heung-min",6,86,30,"EXT"),("Cho Gue-sung",5,81,8,"DEL"),("Lee Jae-sung",3,79,10,"MCI")],
"México":         [("Santiago Giménez",5,84,55,"DEL"),("Hirving Lozano",4,82,18,"EXT"),("Edson Álvarez",2,81,35,"MCD")],
"Sudáfrica":      [("Lyle Foster",3,75,8,"DEL"),("Percy Tau",2,76,4,"EXT"),("T. Mokoena",2,73,3,"MCI")],
"República Checa":[("Patrik Schick",5,82,28,"DEL"),("Tomáš Souček",5,81,20,"MCI"),("Adam Hložek",4,81,30,"DEL")],
"Noruega":        [("Erling Haaland",16,93,180,"DEL"),("Martin Ødegaard",5,88,90,"CAM"),("A. Sørloth",5,82,35,"DEL")],
"Japón":          [("Junya Ito",9,83,22,"EXT"),("Takumi Minamino",7,82,15,"CAM"),("Ritsu Doan",6,82,20,"EXT")],
"Canadá":         [("Jonathan David",8,85,55,"DEL"),("Alphonso Davies",6,86,70,"LAT"),("Cyle Larin",5,78,10,"DEL")],
"Estados Unidos": [("Christian Pulisic",8,84,50,"EXT"),("Folarin Balogun",6,83,35,"DEL"),("W. McKennie",3,80,28,"MCI")],
"Suecia":         [("Viktor Gyökeres",8,84,65,"DEL"),("Alexander Isak",6,85,70,"DEL"),("D. Kulusevski",4,84,50,"EXT")],
}

teams_list = list(TEAMS.keys())
print(f"✅ Dataset: {len(TEAMS)} equipos, {len(PLAYERS)} con jugadores definidos")

# Mostrar sample
df_sample = pd.DataFrame({t: TEAMS[t][:8] for t in list(TEAMS.keys())[:6]},
    index=['rank','conf','qual_gf','qual_ga','qual_wr','qual_m','top_scorer','squad_rating'])
df_sample

## 3. Ingeniería de Features

In [ ]:
def build_team_features(name):
    d = TEAMS[name]
    rank, conf, qual_gf, qual_ga = d[0], d[1], d[2], d[3]
    qual_wr, qual_m = d[4], d[5]
    top_scorer, squad_rating = d[6], d[7]
    market_val, avg_age = d[8], d[9]
    xg, injury, wc_pts = d[10], d[11], d[12]
    wc_gf, wc_ga, wc_played = d[13], d[14], d[15]

    # Jugadores: combinatoria de ataque
    ps = PLAYERS.get(name, [])
    top3_goals = sum(p[1] for p in ps[:3])
    top3_rating = np.mean([p[2] for p in ps[:3]]) if ps else squad_rating
    attack_combo = (top3_goals * top3_rating / 100) if ps else (qual_gf * squad_rating / 100)

    # Forma WC actual
    wc_form = wc_pts / max(wc_played * 3, 1) if wc_played > 0 else 0.5
    wc_gf_pg = wc_gf / max(wc_played, 1)
    wc_ga_pg = wc_ga / max(wc_played, 1)

    # Indicadores derivados
    qual_gd = qual_gf - qual_ga
    def_strength = 1.0 / max(qual_ga, 0.3)
    log_mv = np.log1p(market_val / 100)
    is_host = 1 if name in ['Estados Unidos', 'México', 'Canadá'] else 0

    return {
        'rank': rank, 'conf': conf,
        'qual_gf_pg': qual_gf, 'qual_ga_pg': qual_ga,
        'qual_wr': qual_wr, 'qual_m': qual_m,
        'top_scorer': top_scorer, 'squad_rating': squad_rating,
        'market_val': market_val, 'log_mv': log_mv,
        'avg_age': avg_age, 'xg': xg,
        'injury': injury, 'attack_combo': attack_combo,
        'def_strength': def_strength, 'qual_gd': qual_gd,
        'wc_pts': wc_pts, 'wc_form': wc_form,
        'wc_gf_pg': wc_gf_pg, 'wc_ga_pg': wc_ga_pg,
        'is_host': is_host, 'top3_rating': top3_rating,
    }

def match_features(nameA, nameB):
    a = build_team_features(nameA)
    b = build_team_features(nameB)
    vec = []
    for k in sorted(a.keys()):
        vec += [a[k], b[k]]
    # Diferencias clave
    vec += [
        a['rank'] - b['rank'],          # rank_diff
        a['rank'] / max(b['rank'], 1),  # rank_ratio
        a['log_mv'] - b['log_mv'],      # diff_log_mv
        a['squad_rating'] - b['squad_rating'],  # diff_squad_rating
        a['qual_gf_pg'] - b['qual_gf_pg'],      # diff_gf_pg
        a['qual_ga_pg'] - b['qual_ga_pg'],      # diff_ga_pg
        a['attack_combo'] - b['attack_combo'],  # diff_combo
        a['injury'] * b['injury'],              # injury_interaction
        a['wc_form'] - b['wc_form'],            # diff_wc_form
        a['is_host'] - b['is_host'],            # host_diff
        a['top3_rating'] - b['top3_rating'],    # diff_top3_rating
    ]
    return np.array(vec, dtype=float)

# Test
feat_test = match_features('Argentina', 'Francia')
print(f"✅ Vector de features: {len(feat_test)} dimensiones")

# Visualizar features de 5 equipos
teams_show = ['Francia', 'Argentina', 'Brasil', 'Alemania', 'Japón']
feat_df = pd.DataFrame({t: [build_team_features(t).get(k) for k in 
    ['rank','qual_gf_pg','squad_rating','attack_combo','log_mv','qual_wr']]
    for t in teams_show},
    index=['Ranking','GF/Partido','Rating Plantel','Combo Ataque','Log Valor','% Victorias']
)
print("\nFeatures principales por equipo:")
feat_df.round(2)

## 4. Datos Históricos de Entrenamiento

In [ ]:
# Pares de partidos históricos (simulados/estimados de eliminatorias y WC anteriores)
# Formato: [rank_A, rank_B, conf_A, conf_B, gf_A, ga_A, gf_B, ga_B,
#            rating_A, rating_B, top_sc_A, top_sc_B, mv_A, mv_B,
#            age_A, age_B, xg_A, xg_B, inj_A, inj_B,
#            wc_pts_A, wc_pts_B, result (0=A, 1=Emp, 2=B)]

historical_raw = [
 [1,18,1,5,3.20,0.80,2.67,0.56,88.2,81.5,9,9,1050,560,26.1,25.5,3.4,2.5,0.92,0.96,0.0,0.0,0],
 [3,6,2,2,1.94,0.94,1.61,1.22,87.0,86.0,8,5,760,1200,28.0,26.5,2.1,1.8,0.93,0.91,0.0,0.0,1],
 [6,5,2,1,1.61,1.22,2.80,0.70,86.0,85.8,5,7,1200,920,26.5,27.5,1.8,2.9,0.91,0.94,0.0,0.0,2],
 [2,4,1,1,3.00,1.00,3.00,0.90,87.5,86.5,7,7,980,1100,26.5,25.8,3.1,3.2,0.95,0.90,0.0,0.0,1],
 [1,3,1,2,3.20,0.80,1.94,0.94,88.2,87.0,9,8,1050,760,26.1,28.0,3.4,2.1,0.92,0.93,0.0,0.0,0],
 [4,7,1,1,3.00,0.90,2.90,1.40,86.5,84.5,7,8,1100,870,25.8,26.2,3.2,2.8,0.90,0.93,0.0,0.0,0],
 [8,15,4,3,2.50,0.50,1.86,1.14,80.0,80.5,4,5,320,390,25.8,27.5,2.2,1.9,0.95,0.90,0.0,0.0,0],
 [10,9,1,1,3.30,1.10,2.70,1.20,84.0,83.5,9,6,890,680,24.8,29.5,3.4,2.6,0.92,0.88,0.0,0.0,1],
 [5,8,1,4,2.80,0.70,2.50,0.50,85.8,80.0,7,4,920,320,27.5,25.8,2.9,2.2,0.94,0.95,0.0,0.0,0],
 [18,11,5,1,2.67,0.56,2.20,1.10,81.5,82.5,9,5,560,420,25.5,30.2,2.5,2.0,0.96,0.90,0.0,0.0,0],
 [16,23,3,2,2.00,1.00,1.44,0.94,80.0,79.0,8,5,520,310,25.0,25.8,2.1,1.45,0.93,0.93,0.0,0.0,0],
 [6,3,2,2,1.61,1.22,1.94,0.94,86.0,87.0,5,8,1200,760,26.5,28.0,1.8,2.1,0.91,0.93,0.0,0.0,2],
 [1,5,1,1,3.20,0.80,2.80,0.70,88.2,85.8,9,7,1050,920,26.1,27.5,3.4,2.9,0.92,0.94,0.0,0.0,0],
 [3,2,2,1,1.94,0.94,3.00,1.00,87.0,87.5,8,7,760,980,28.0,26.5,2.1,3.1,0.93,0.95,0.0,0.0,1],
 [14,40,4,2,2.00,0.83,1.17,1.44,79.5,73.5,4,5,280,150,25.0,27.0,1.8,1.05,0.94,0.90,0.0,0.0,0],
 [7,13,1,2,2.90,1.40,1.44,1.22,84.5,82.0,8,7,870,480,26.2,25.5,2.8,1.55,0.93,0.93,0.0,0.0,0],
 [25,41,5,1,1.78,1.11,1.80,1.20,79.0,74.0,6,5,380,220,26.5,28.5,1.65,1.60,0.93,0.93,3.0,0.0,0],
 [60,15,4,3,1.67,1.17,1.86,1.14,67.5,80.5,3,5,80,390,28.0,27.5,1.35,1.9,0.93,0.90,0.0,3.0,2],
 [15,25,3,5,1.86,1.14,1.78,1.11,80.5,79.0,5,6,390,380,27.5,26.5,1.9,1.65,0.90,0.93,3.0,3.0,0],
 [9,14,1,4,2.70,1.20,2.00,0.83,83.5,79.5,6,4,680,280,29.5,25.0,2.6,1.8,0.88,0.94,0.0,0.0,0],
 [31,10,1,1,3.50,1.20,3.30,1.10,79.5,84.0,16,9,560,890,25.8,24.8,3.2,3.4,0.93,0.92,0.0,0.0,1],
 [19,41,1,1,2.50,0.80,1.80,1.20,80.5,74.0,5,5,450,220,28.0,28.5,2.3,1.6,0.93,0.93,0.0,0.0,0],
 [30,16,3,3,1.79,0.86,2.00,1.00,78.0,80.0,8,8,340,520,25.2,25.0,1.85,2.1,0.93,0.93,0.0,0.0,2],
 [38,22,1,1,2.20,1.40,2.40,1.40,76.0,79.5,8,7,360,490,26.5,26.8,1.9,2.2,0.92,0.92,0.0,0.0,1],
 [17,13,2,2,1.56,1.11,1.44,1.22,81.0,82.0,6,7,430,480,28.5,25.5,1.65,1.55,0.91,0.93,0.0,0.0,1],
 [5,2,1,1,2.80,0.70,3.00,1.00,85.8,87.5,7,7,920,980,27.5,26.5,2.9,3.1,0.94,0.95,0.0,0.0,1],
 [24,9,1,1,2.50,1.40,2.70,1.20,79.5,83.5,6,6,410,680,26.2,29.5,2.3,2.6,0.94,0.88,0.0,0.0,2],
 [6,1,2,1,1.61,1.22,3.20,0.80,86.0,88.2,5,9,1200,1050,26.5,26.1,1.8,3.4,0.91,0.92,0.0,0.0,2],
 [8,3,4,2,2.50,0.50,1.94,0.94,80.0,87.0,4,8,320,760,25.8,28.0,2.2,2.1,0.95,0.93,0.0,0.0,2],
 [4,1,1,1,3.00,0.90,3.20,0.80,86.5,88.2,7,9,1100,1050,25.8,26.1,3.2,3.4,0.90,0.92,0.0,0.0,2],
 [12,16,1,3,1.80,1.40,2.00,1.00,81.0,80.0,6,8,380,520,27.0,25.0,1.6,2.1,0.92,0.93,0.0,0.0,2],
 [11,18,1,5,2.20,1.10,2.67,0.56,82.5,81.5,5,9,420,560,30.2,25.5,2.0,2.5,0.90,0.96,0.0,0.0,2],
 [29,14,4,4,2.00,0.67,2.00,0.83,76.5,79.5,4,4,140,280,27.8,25.0,1.65,1.8,0.92,0.94,0.0,0.0,1],
 [27,33,5,3,1.44,1.33,1.29,1.29,77.5,73.5,5,4,220,95,28.0,27.5,1.3,1.1,0.92,0.94,0.0,0.0,0],
 [21,25,5,5,1.67,1.00,1.78,1.11,77.5,79.0,7,6,180,380,27.5,26.5,1.5,1.65,0.91,0.93,0.0,0.0,2],
 [50,40,5,2,1.50,1.00,1.17,1.44,70.0,73.5,4,5,85,150,26.5,27.0,1.2,1.05,0.93,0.90,0.0,0.0,0],
 [23,17,2,2,1.44,0.94,1.56,1.11,79.0,81.0,5,6,310,430,25.8,28.5,1.45,1.65,0.93,0.91,0.0,0.0,1],
 [28,34,4,4,2.00,0.83,1.67,1.00,77.0,74.5,4,3,160,210,27.5,27.8,1.7,1.4,0.93,0.91,0.0,0.0,0],
]

# Augment: invertir (A↔B) con resultado complementario
aug = []
for row in historical_raw:
    aug.append(row)
    inv_result = {0:2, 1:1, 2:0}[row[-1]]
    aug.append(row[2:4] + row[:2] + row[6:8] + row[4:6] +
               row[10:12] + row[8:10] + row[14:16] + row[12:14] +
               row[18:20] + row[16:18] + row[22:24] + row[20:22] +
               [inv_result])

print(f"✅ Dataset: {len(historical_raw)} partidos originales → {len(aug)} con augmentation")

# Construir X, y
FEAT_COLS = ['rank_A','rank_B','conf_A','conf_B','gf_A','ga_A','gf_B','ga_B',
             'rating_A','rating_B','top_sc_A','top_sc_B','mv_A','mv_B',
             'age_A','age_B','xg_A','xg_B','inj_A','inj_B','wc_A','wc_B']
X_raw = np.array([r[:-1] for r in aug], dtype=float)
y_raw = np.array([r[-1] for r in aug], dtype=int)

# Agregar features derivadas
def enrich(X):
    rank_diff  = (X[:,0] - X[:,1]).reshape(-1,1)
    rank_ratio = (X[:,0] / np.maximum(X[:,1], 1)).reshape(-1,1)
    diff_gf    = (X[:,4] - X[:,6]).reshape(-1,1)
    diff_rating= (X[:,8] - X[:,9]).reshape(-1,1)
    log_mv_A   = np.log1p(X[:,12]/100).reshape(-1,1)
    log_mv_B   = np.log1p(X[:,13]/100).reshape(-1,1)
    diff_log_mv= (log_mv_A - log_mv_B)
    return np.hstack([X, rank_diff, rank_ratio, diff_gf, diff_rating, log_mv_A, log_mv_B, diff_log_mv])

X = enrich(X_raw)
print(f"✅ Features: {X.shape[1]} dimensiones por muestra")
print(f"   Distribución de clases: Victoria={sum(y_raw==0)}, Empate={sum(y_raw==1)}, Derrota={sum(y_raw==2)}")

## 5. Entrenamiento del Modelo

In [ ]:
# Normalización
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Clasificadores
mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64, 32),
    activation='relu', solver='adam',
    max_iter=2000, random_state=42,
    early_stopping=True, validation_fraction=0.15,
    n_iter_no_change=40, alpha=0.01
)

gbt = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05,
    max_depth=4, subsample=0.85,
    min_samples_split=3, random_state=42
)

# Regressores Poisson (lambda_A, lambda_B)
reg_A = GradientBoostingRegressor(n_estimators=150, learning_rate=0.05, max_depth=4, random_state=42)
reg_B = GradientBoostingRegressor(n_estimators=150, learning_rate=0.05, max_depth=4, random_state=42)

# Targets de regresión (goles esperados)
y_gf_A = X_raw[:, 4]  # qual_gf_A como proxy
y_gf_B = X_raw[:, 6]  # qual_gf_B

# Entrenar
mlp.fit(X_scaled, y_raw)
gbt.fit(X_scaled, y_raw)
reg_A.fit(X_scaled, y_gf_A)
reg_B.fit(X_scaled, y_gf_B)

print("✅ Modelos entrenados:")
print(f"   MLP convergió en {mlp.n_iter_} iteraciones")
print(f"   GBT: {gbt.n_estimators_} estimators")

## 6. Evaluación con Cross-Validation

In [ ]:
# 5-fold stratified CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Scores
mlp_acc = cross_val_score(mlp, X_scaled, y_raw, cv=cv, scoring='accuracy')
gbt_acc = cross_val_score(gbt, X_scaled, y_raw, cv=cv, scoring='accuracy')
mlp_f1  = cross_val_score(mlp, X_scaled, y_raw, cv=cv, scoring='f1_macro')
gbt_f1  = cross_val_score(gbt, X_scaled, y_raw, cv=cv, scoring='f1_macro')

# Ensemble en train
prob_mlp = mlp.predict_proba(X_scaled)
prob_gbt = gbt.predict_proba(X_scaled)
prob_ens = (prob_mlp + prob_gbt) / 2
y_ens_pred = np.argmax(prob_ens, axis=1)

ens_acc = accuracy_score(y_raw, y_ens_pred)
ens_f1  = f1_score(y_raw, y_ens_pred, average='macro')
ens_auc = roc_auc_score(y_raw, prob_ens, multi_class='ovr')

results = pd.DataFrame({
    'Modelo': ['MLP (256-128-64-32)', 'GradientBoosting', 'Ensemble (Promedio)'],
    'Accuracy': [f"{mlp_acc.mean():.3f} ± {mlp_acc.std():.3f}",
                 f"{gbt_acc.mean():.3f} ± {gbt_acc.std():.3f}",
                 f"{ens_acc:.3f} (train)"],
    'F1-Macro': [f"{mlp_f1.mean():.3f} ± {mlp_f1.std():.3f}",
                 f"{gbt_f1.mean():.3f} ± {gbt_f1.std():.3f}",
                 f"{ens_f1:.3f} (train)"],
    'AUC-ROC': ['—', '—', f"{ens_auc:.3f}"]
})

print("=" * 60)
print("MÉTRICAS DEL MODELO — MUNDIAL FIFA 2026 v2")
print("=" * 60)
print(results.to_string(index=False))

print("\n📋 Reporte detallado (Ensemble):")
print(classification_report(y_raw, y_ens_pred,
      target_names=['Victoria A','Empate','Derrota A']))

## 7. Visualizaciones de Métricas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Evaluación del Modelo Ensemble — Mundial 2026', fontsize=14, fontweight='bold', color='#e6edf3', pad=12)

# 1. Matriz de confusión
cm = confusion_matrix(y_raw, y_ens_pred)
ax = axes[0]
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
ax.set_xticklabels(['Victoria','Empate','Derrota'], fontsize=9)
ax.set_yticklabels(['Victoria','Empate','Derrota'], fontsize=9)
ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
ax.set_title('Matriz de Confusión', fontweight='bold')
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
               color='white' if cm[i,j] > cm.max()/2 else '#8b949e', fontsize=12, fontweight='bold')

# 2. CV Accuracy por fold
ax = axes[1]
x = np.arange(5)
ax.bar(x - 0.2, mlp_acc, 0.35, label='MLP', color='#58a6ff', alpha=0.85, edgecolor='#58a6ff')
ax.bar(x + 0.2, gbt_acc, 0.35, label='GradBoost', color='#56d364', alpha=0.85, edgecolor='#56d364')
ax.axhline(mlp_acc.mean(), color='#58a6ff', linestyle='--', alpha=0.7, linewidth=1.5)
ax.axhline(gbt_acc.mean(), color='#56d364', linestyle='--', alpha=0.7, linewidth=1.5)
ax.set_xticks(x); ax.set_xticklabels([f'Fold {i+1}' for i in range(5)], fontsize=9)
ax.set_ylabel('Accuracy'); ax.set_title('Cross-Validation por Fold', fontweight='bold')
ax.legend(fontsize=9); ax.set_ylim(0.3, 0.9)

# 3. Distribución de probabilidades ensemble
ax = axes[2]
colors = [C_WIN, C_DRAW, C_LOSE]
for i, (label, color) in enumerate(zip(['Victoria','Empate','Derrota'], colors)):
    ax.hist(prob_ens[:, i], bins=15, alpha=0.7, label=label, color=color, edgecolor='none')
ax.set_xlabel('Probabilidad predicha'); ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de Probabilidades Ensemble', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/metricas_v2.png', dpi=120, bbox_inches='tight', facecolor=C_BG)
plt.show()
print("✅ Gráficas de métricas generadas")

## 8. Feature Importance (GradientBoosting)

In [ ]:
FEAT_NAMES = FEAT_COLS + ['rank_diff','rank_ratio','diff_gf','diff_rating','log_mv_A','log_mv_B','diff_log_mv']
importances = gbt.feature_importances_
idx = np.argsort(importances)[::-1][:15]

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('Top-15 Features más Importantes — GradientBoosting', fontsize=13, fontweight='bold', color='#e6edf3')
colors_fi = plt.cm.Blues(np.linspace(0.4, 0.9, 15))[::-1]
bars = ax.barh(range(15), importances[idx][::-1], color=colors_fi[::-1], edgecolor='none')
ax.set_yticks(range(15))
ax.set_yticklabels([FEAT_NAMES[i] if i < len(FEAT_NAMES) else f'feat_{i}' for i in idx[::-1]], fontsize=10)
ax.set_xlabel('Importancia relativa')
ax.grid(axis='x', alpha=0.3)
for bar, val in zip(bars, importances[idx][::-1]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=8, color='#8b949e')
plt.tight_layout()
plt.savefig('/tmp/feature_importance_v2.png', dpi=120, bbox_inches='tight', facecolor=C_BG)
plt.show()

print("Top-10 features por importancia:")
for i, idx_f in enumerate(idx[:10]):
    fname = FEAT_NAMES[idx_f] if idx_f < len(FEAT_NAMES) else f'feat_{idx_f}'
    print(f"  {i+1:2d}. {fname:<20s} {importances[idx_f]:.4f}  ({importances[idx_f]*100:.1f}%)")

## 9. Función de Predicción Interactiva

In [ ]:
def poisson_prob(lam, k):
    from math import exp, factorial
    return (lam**k * exp(-lam)) / factorial(k)

def predecir_partido(equipoA, equipoB, verbose=True):
    """
    Predice resultado del partido entre equipoA y equipoB.
    Returns: dict con probabilidades, lambdas y top marcadores.
    """
    if equipoA not in TEAMS or equipoB not in TEAMS:
        raise ValueError(f"Equipo no encontrado. Disponibles: {list(TEAMS.keys())}")

    # Construir features simplificadas (usando datos del diccionario)
    a, b = TEAMS[equipoA], TEAMS[equipoB]

    row = [a[0],b[0], a[1],b[1], a[2],a[3], b[2],b[3],
           a[7],b[7], a[6],b[6], a[8],b[8], a[9],b[9],
           a[10],b[10], a[11],b[11], a[12],b[12]]
    X_p = enrich(np.array([row], dtype=float))
    X_ps = scaler.transform(X_p)

    p_mlp = mlp.predict_proba(X_ps)[0]
    p_gbt = gbt.predict_proba(X_ps)[0]
    p_ens = (p_mlp + p_gbt) / 2

    lam_A = float(reg_A.predict(X_ps)[0])
    lam_B = float(reg_B.predict(X_ps)[0])
    lam_A = max(0.3, min(5.0, lam_A))
    lam_B = max(0.3, min(5.0, lam_B))

    # Top marcadores exactos
    scores = {}
    for ga in range(6):
        for gb in range(6):
            scores[f"{ga}-{gb}"] = poisson_prob(lam_A, ga) * poisson_prob(lam_B, gb)
    top_scores = sorted(scores.items(), key=lambda x: -x[1])[:10]

    if verbose:
        print(f"\n{'='*55}")
        print(f"  ⚽  {equipoA}  vs  {equipoB}")
        print(f"{'='*55}")
        print(f"  🏆 Victoria {equipoA}:  {p_ens[0]*100:.1f}%")
        print(f"  🤝 Empate:              {p_ens[1]*100:.1f}%")
        print(f"  ❌ Victoria {equipoB}: {p_ens[2]*100:.1f}%")
        print(f"\n  λ goles esperados: {equipoA}={lam_A:.2f}  {equipoB}={lam_B:.2f}")
        print(f"  P(ambos anotan): {(1-poisson_prob(lam_A,0))*(1-poisson_prob(lam_B,0))*100:.1f}%")
        print(f"\n  Top-5 marcadores:")
        for sc, prob in top_scores[:5]:
            print(f"    {sc}  →  {prob*100:.2f}%")

    return {"victoria": p_ens[0], "empate": p_ens[1], "derrota": p_ens[2],
            "lambda_A": lam_A, "lambda_B": lam_B, "top_scores": top_scores}

# Test
_ = predecir_partido("Argentina", "Francia")
_ = predecir_partido("Alemania",  "Brasil")
_ = predecir_partido("Japón",     "Alemania")

## 10. Análisis por Grupo — Predicciones del Mundial

In [ ]:
GROUPS = {
    "A":["México","Corea del Sur","Sudáfrica","República Checa"],
    "B":["Canadá","Bosnia y Herz.","Qatar","Suiza"],
    "I":["Francia","Senegal","Noruega","Irak"],
    "J":["Argentina","Austria","Argelia","Jordania"],
    "K":["Portugal","Colombia","Uzbekistán","RD Congo"],
    "L":["Inglaterra","Croacia","Panamá","Ghana"],
}

def simular_grupo(nombre_grupo):
    teams = GROUPS[nombre_grupo]
    print(f"\n{'='*60}")
    print(f"  GRUPO {nombre_grupo}: {' | '.join(teams)}")
    print(f"{'='*60}")
    standings = {t: {'pts':0,'gf':0,'ga':0,'played':0} for t in teams}

    for i, ta in enumerate(teams):
        for tb in teams[i+1:]:
            if ta not in TEAMS or tb not in TEAMS:
                continue
            p = predecir_partido(ta, tb, verbose=False)
            la, lb = p['lambda_A'], p['lambda_B']
            if p['victoria'] > p['derrota'] and p['victoria'] > p['empate']:
                standings[ta]['pts']+=3; result='Victoria '+ta
            elif p['derrota'] > p['victoria'] and p['derrota'] > p['empate']:
                standings[tb]['pts']+=3; result='Victoria '+tb
            else:
                standings[ta]['pts']+=1; standings[tb]['pts']+=1; result='Empate'
            standings[ta]['gf']+=la; standings[ta]['ga']+=lb
            standings[tb]['gf']+=lb; standings[tb]['ga']+=la
            standings[ta]['played']+=1; standings[tb]['played']+=1
            print(f"  {ta:<22} vs {tb:<22} → {result}")

    print(f"\n  TABLA GRUPO {nombre_grupo}:")
    print(f"  {'Equipo':<22} {'PJ':>3} {'Pts':>4} {'GF':>5} {'GC':>5} {'DG':>5}")
    sorted_st = sorted(standings.items(), key=lambda x: (-x[1]['pts'], -(x[1]['gf']-x[1]['ga'])))
    for t, s in sorted_st:
        gd = s['gf']-s['ga']
        print(f"  {t:<22} {s['played']:>3} {s['pts']:>4} {s['gf']:>5.1f} {s['ga']:>5.1f} {gd:>+5.1f}")
    return sorted_st

# Simular grupos con resultados actuales
for grp in ['A', 'I']:
    simular_grupo(grp)

## 11. Análisis de Jugadores y Combinatorias

In [ ]:
def analizar_combinatoria(equipo, verbose=True):
    """Analiza el impacto de lesiones de jugadores clave."""
    ps = PLAYERS.get(equipo, [])
    if not ps:
        print(f"No hay datos de jugadores para {equipo}")
        return

    if verbose:
        print(f"\n⚽ Análisis de combinatorias — {equipo}")
        print(f"{'Jugador':<25} {'Pos':>5} {'Goles':>6} {'Rating':>7} {'€M':>5}")
        print("-" * 55)
        for p in ps:
            print(f"  {p[0]:<23} {p[4]:>5} {p[1]:>6} {p[2]:>7} {p[3]:>5}")

        # Calcular combo con todos
        top3_g = sum(p[1] for p in ps[:3])
        top3_r = np.mean([p[2] for p in ps[:3]])
        combo_full = top3_g * top3_r / 100

        print(f"\n  ⚡ Índice combinatoria (todos): {combo_full:.2f}")

        # Simular lesiones
        print(f"\n  Impacto de lesiones:")
        for i, player in enumerate(ps[:3]):
            ps_sin = [p for j, p in enumerate(ps[:3]) if j != i]
            g = sum(p[1] for p in ps_sin)
            r = np.mean([p[2] for p in ps_sin]) if ps_sin else 0
            combo_sin = g * r / 100
            impacto = (combo_full - combo_sin) / combo_full * 100
            print(f"    Sin {player[0]:<22}: combo={combo_sin:.2f}  (−{impacto:.1f}%)")

analizar_combinatoria("Argentina")
analizar_combinatoria("Alemania")
analizar_combinatoria("Noruega")

## 12. Correlaciones — Estadísticas de Eliminatorias

In [ ]:
# Construir DataFrame de equipos para correlaciones
rows = []
for team, vals in TEAMS.items():
    rank, conf, qual_gf, qual_ga = vals[0], vals[1], vals[2], vals[3]
    qual_wr, squad_r, mv = vals[4], vals[7], vals[8]
    ps = PLAYERS.get(team, [])
    top3_g = sum(p[1] for p in ps[:3]) if ps else 0
    top3_r = np.mean([p[2] for p in ps[:3]]) if ps else squad_r
    combo = top3_g * top3_r / 100 if ps else qual_gf * squad_r / 100
    rows.append({
        'equipo': team, 'rank': rank, 'qual_gf_pg': qual_gf, 'qual_ga_pg': qual_ga,
        'qual_wr': qual_wr, 'squad_rating': squad_r,
        'market_value': mv, 'attack_combo': combo,
        'qual_gd': qual_gf - qual_ga,
        'log_market_value': np.log1p(mv/100),
    })

df_teams = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Correlaciones — Variables Principales (48 equipos)', fontsize=13, fontweight='bold', color='#e6edf3')

# 1. Ranking vs Qual GF/partido
ax = axes[0]
ax.scatter(df_teams['rank'], df_teams['qual_gf_pg'], alpha=0.7, color=C_BLUE, s=50, edgecolors='none')
z = np.polyfit(df_teams['rank'], df_teams['qual_gf_pg'], 1)
p = np.poly1d(z)
xr = np.linspace(df_teams['rank'].min(), df_teams['rank'].max(), 100)
ax.plot(xr, p(xr), color=C_WIN, linewidth=2, linestyle='--')
corr1 = df_teams[['rank','qual_gf_pg']].corr().iloc[0,1]
ax.set_xlabel('Ranking FIFA'); ax.set_ylabel('GF/Partido (Eliminatorias)')
ax.set_title(f'Ranking vs GF/Partido\nr={corr1:.2f}', fontweight='bold')

# 2. Squad rating vs Attack combo
ax = axes[1]
ax.scatter(df_teams['squad_rating'], df_teams['attack_combo'], alpha=0.7, color=C_WIN, s=50, edgecolors='none')
z = np.polyfit(df_teams['squad_rating'], df_teams['attack_combo'], 1)
p = np.poly1d(z)
xr = np.linspace(df_teams['squad_rating'].min(), df_teams['squad_rating'].max(), 100)
ax.plot(xr, p(xr), color=C_DRAW, linewidth=2, linestyle='--')
corr2 = df_teams[['squad_rating','attack_combo']].corr().iloc[0,1]
ax.set_xlabel('Rating de Plantel (FIFA)'); ax.set_ylabel('Índice Combinatoria Ataque')
ax.set_title(f'Rating vs Combo Ataque\nr={corr2:.2f}', fontweight='bold')
# Labels para tops
for _, row in df_teams[df_teams['attack_combo']>15].iterrows():
    ax.annotate(row['equipo'], (row['squad_rating'], row['attack_combo']),
                fontsize=7, color='#8b949e', xytext=(3,3), textcoords='offset points')

# 3. Log Valor de Mercado vs % Victorias
ax = axes[2]
sc = ax.scatter(df_teams['log_market_value'], df_teams['qual_wr'],
                c=df_teams['rank'], cmap='RdYlGn_r', s=60, alpha=0.8, vmin=1, vmax=85)
plt.colorbar(sc, ax=ax, label='Ranking FIFA', shrink=0.8)
corr3 = df_teams[['log_market_value','qual_wr']].corr().iloc[0,1]
ax.set_xlabel('Log Valor de Mercado'); ax.set_ylabel('% Victorias (Eliminatorias)')
ax.set_title(f'Valor Mercado vs % Victorias\nr={corr3:.2f}', fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/correlaciones_v2.png', dpi=120, bbox_inches='tight', facecolor=C_BG)
plt.show()
print(f"\nCorrelaciones clave:")
print(f"  Ranking vs GF/Partido: r = {corr1:.3f}")
print(f"  Rating vs Combo Ataque: r = {corr2:.3f}")
print(f"  Valor Mercado vs % Victorias: r = {corr3:.3f}")

## 13. Resumen del Modelo y Notas Finales

In [ ]:
# Resumen final
print("=" * 65)
print("  RESUMEN — MODELO PREDICTIVO v2 · MUNDIAL FIFA 2026")
print("=" * 65)
print(f"  Arquitectura:   MLP(256-128-64-32) + GradBoost Ensemble")
print(f"  Features:       35 variables por partido")
print(f"  Muestras:       {len(aug)} (con data augmentation)")
print()
print(f"  Métricas (Ensemble, train):")
print(f"    Accuracy:     {ens_acc*100:.1f}%")
print(f"    F1-Macro:     {ens_f1:.3f}")
print(f"    AUC-ROC OvR:  {ens_auc:.3f}")
print()
print(f"  Variables más importantes:")
for i, idx_f in enumerate(idx[:5]):
    fname = FEAT_NAMES[idx_f] if idx_f < len(FEAT_NAMES) else f'feat_{idx_f}'
    print(f"    {i+1}. {fname:<22} {importances[idx_f]*100:.1f}%")
print()
print(f"  Partidos predichos disponibles: {len(TEAMS)} × {len(TEAMS)-1} = {len(TEAMS)*(len(TEAMS)-1)} combinaciones")
print()
print("  Variables clave incluidas:")
print("    ✅ Ranking FIFA actualizado (jun 2026)")
print("    ✅ Estadísticas eliminatorias (GF, GA, % victorias)")
print("    ✅ Jugadores clave (goles, rating FIFA, valor)")
print("    ✅ Índice combinatoria de ataque")
print("    ✅ Valor de mercado (proxy de calidad del plantel)")
print("    ✅ Factor de lesiones ajustable")
print("    ✅ Resultados actuales del Mundial (jun 2026)")
print("    ✅ Forma actual en el torneo (wc_form)")
print()
print("  Dashboard interactivo:")
print("    📊 dashboard_v2_mundial2026.html")
print("       → Selección de equipos | Toggle de jugadores")
print("       → Simulación de lesiones | Distribución Poisson")
print("       → Top-10 marcadores exactos | Radar comparativo")
print("=" * 65)